# Predicting Credit Default Risk Using Machine Learning on the Home Credit Dataset

## Introduction
Financial institutions face challenges in assessing the creditworthiness of loan applicants, especially those with limited credit history. Traditional scoring methods often fail to capture complex behavioral patterns, leading to increased default rates or missed opportunities. 

This project aims to develop a data-driven machine learning model to accurately predict the likelihood of loan default, using demographic, financial, and behavioral data from the Home Credit Default Risk dataset.


## Objectives
- Investigate the predictive power of various machine learning techniques on credit risk.
- Engineer meaningful features to improve model accuracy and interpretability.
- Compare traditional models with advanced ensemble methods.
- Deliver actionable insights for financial institutions to make better lending decisions.

In [ ]:
# Import all the required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix
from imblearn.over_sampling import SMOTE
import shap
import os
import warnings

## Data Collection
The dataset used for this project is the Home Credit Default Risk dataset from Kaggle. It includes:
- Client application data (demographics, employment, income).
- Credit bureau and previous loan information.
- Behavioral patterns from installment payments and credit card balances.

All data is anonymized and pre-collected, eliminating the need for external scraping or manual collection.

In [ ]:
current_dir = os.getcwd()

# Load the dataset
file_path = "credit_risk_dataset.csv"  # Replace with your dataset path
df = pd.read_csv(file_path)

### Data Preprocessing
Steps taken to prepare the data:
1. **Handling Missing Values:** Missing numerical values were imputed using the median, while categorical values were filled with the mode.
2. **Outlier Detection:** Outliers were identified using statistical methods (e.g., Z-score or IQR) but not explicitly removed due to their potential importance in predictive modeling.
3. **Feature Engineering:** Created new features such as `debt-to-income ratio` to capture financial risk.

In [ ]:
# Display the first few rows
print("First few rows of the dataset:")
df.head()

In [ ]:
df.info()

In [ ]:
# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Handle missing values: Impute numerical features with median
num_cols = df.select_dtypes(include=np.number).columns
imputer_num = SimpleImputer(strategy='median')
df[num_cols] = imputer_num.fit_transform(df[num_cols])

# Handle missing values: Impute categorical features with mode
cat_cols = df.select_dtypes(include='object').columns
imputer_cat = SimpleImputer(strategy='most_frequent')
df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])

# Encode categorical variables
print("\nUnique values before encoding:")
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")

In [ ]:
df.isnull().sum()

In [ ]:
# Convert 'cb_person_default_on_file' (Y/N) to binary (0/1)
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'Y': 1, 'N': 0})

# One-hot encode other categorical columns
df = pd.get_dummies(df, columns=['person_home_ownership', 'loan_intent', 'loan_grade'], drop_first=True)

print("\nFirst few rows after preprocessing:")
print(df.head())

### Exploratory Data Analysis
Key insights from EDA:
1. **Statistical Summaries:** Descriptive statistics revealed trends in income levels, loan amounts, and employment lengths.
2. **Correlation Heatmaps:** Identified relationships between features like `loan_amnt`, `person_income`, and `loan_status`.
3. **Distribution Plots:** Visualized distributions of key features (e.g., income, loan amount) to understand their impact on default risk.

In [ ]:
# Select only numeric columns for correlation
numeric_df = df.select_dtypes(include=['number'])

# Plot correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

#Summarize findings from statistical summaries, heatmaps, and distribution plots.

In [ ]:
# Ignore the specific FutureWarnings
warnings.filterwarnings("ignore", category=FutureWarning) 

# Feature Engineering: Debt-to-Income Ratio
df['debt_to_income'] = df['loan_amnt'] / df['person_income']

# Split data into features and target
X = df.drop(columns=['loan_status'])
y = df['loan_status']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize numerical features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Handle class imbalance using SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)


**The distribution of loan interest rates is right-skewed, with most values between 10% and 20%, and some loans reaching above 25%.**


In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['loan_int_rate'].dropna(), bins=30)
plt.title("Distribution of Loan Interest Rate")
plt.xlabel("Interest Rate (%)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
df = pd.read_csv("credit_risk_dataset.csv")

# Count plot: Loan Intent
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='loan_intent', order=df['loan_intent'].value_counts().index)
plt.title("Distribution of Loan Intent")
plt.xlabel("Loan Purpose")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


### Model Building
Models implemented:
1. **Baseline Models:**
   - Logistic Regression: Provides a simple linear baseline for comparison.
   - Decision Tree: Captures non-linear relationships between features.

2. **Advanced Models:**
   - Random Forest: An ensemble method that improves accuracy by aggregating multiple decision trees.

Future Work:
- Implement XGBoost or LightGBM for further performance improvements.
- Explore deep learning models if time allows.

In [ ]:
# --- Model Building ---
# Logistic Regression
lr = LogisticRegression(random_state=42, solver='liblinear')
lr.fit(X_train_resampled, y_train_resampled)
y_pred_lr = lr.predict(X_test)

# Decision Tree
dt = DecisionTreeClassifier(random_state=42, max_depth=5)
dt.fit(X_train_resampled, y_train_resampled)
y_pred_dt = dt.predict(X_test)

rf = RandomForestClassifier(random_state=42, n_estimators=100)
rf.fit(X_train_resampled, y_train_resampled)
y_pred_rf = rf.predict(X_test)


### Evaluation Metrics
Metrics used to evaluate model performance:
1. **ROC-AUC Score:** Measures how well models distinguish between defaulted and non-defaulted loans.
2. **Precision & Recall:** Evaluates accuracy in predicting defaults while minimizing false positives/negatives.
3. **F1-score:** Balances precision and recall into a single metric.
4. **Confusion Matrix:** Visualizes prediction results for further analysis.

In [ ]:
## --- Model Evaluation ---
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    print(f"--- {model_name} ---")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

    # ROC AUC Score
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    print(f'ROC AUC Score: {auc}')

    # ROC Curve
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {model_name}')
    plt.legend()
    plt.show()
evaluate_model(lr, X_test, y_test, "Logistic Regression")
evaluate_model(dt, X_test, y_test, "Decision Tree")
evaluate_model(rf, X_test, y_test, "Random Forest")

### Model Interpretability
Tools used:
1. **SHAP (SHapley Additive exPlanations):**
   - Visualized feature importance using SHAP summary plots.
   - Explained individual predictions using SHAP force plots.

2. **Feature Importance Analysis:**
   - Ranked features by their mean absolute SHAP values to identify key predictors of default risk.

Future Work:
- Explore Grad-CAM if deep learning models are implemented later.

In [ ]:
# 1. Sample the data
sample_size = 100
sample_indices = np.random.choice(X_test.shape[0], sample_size, replace=False)
X_sample = X_test[sample_indices]
X_sample_df = pd.DataFrame(X_sample, columns=X.columns)

# 2. SHAP Explainer
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_sample)

# 3. Handle multiclass (binary) output
if isinstance(shap_values, list):
    shap_values = shap_values[1]

# 4. SHAP summary plot
shap.summary_plot(shap_values, X_sample_df, feature_names=X.columns)

# 5. Calculate feature importance (for RandomForest)
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
})
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

# 6. Explanations for top features
print("\nDetailed explanations for top features:")
for feature in feature_importance['Feature'].head(5):
    print(f"\nFeature: {feature}")
    print(f"- High SHAP values for {feature} indicate a strong impact on the model's prediction.")
    print(f"- Direction of SHAP value tells whether it pushes the prediction towards default or not.")


In [ ]:
# Fill missing values in 'loan_int_rate' with the median
df['loan_int_rate'] = df['loan_int_rate'].fillna(df['loan_int_rate'].median())

# Drop rows with missing 'person_emp_length' if necessary
df = df.dropna(subset=['person_emp_length'])

# Convert 'cb_person_default_on_file' (Y/N) to binary (1/0)
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'Y': 1, 'N': 0})

# One-hot encode 'person_home_ownership', 'loan_intent', and 'loan_grade'
df = pd.get_dummies(df, columns=['person_home_ownership', 'loan_intent', 'loan_grade'], drop_first=True)

# Define features and target variable
X = df.drop(columns=['loan_status'])
y = df['loan_status']

# Split into training and testing sets (80% train, 20% test)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model's performance
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))


## Results & Discussion
Key findings:
1. Random Forest outperformed baseline models (Logistic Regression and Decision Tree) across all metrics (e.g., ROC-AUC Score).
2. SHAP analysis revealed that `person_income`, `loan_amnt`, and `debt_to_income` were critical predictors of default risk.

Limitations:
- Class imbalance affected model performance; SMOTE was used to address this issue.
- Outlier removal was not implemented due to time constraints but could improve accuracy further.

Recommendations:
- Use Random Forest for production deployment due to its robust performance and interpretability via SHAP analysis.


In [ ]:
#Step 5: Feature Importance Analysis

# Feature importance plot
importances = rf.feature_importances_
feature_names = X.columns

plt.figure(figsize=(15, 8))
sns.barplot(x=importances, y=feature_names)
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Conclusion
This project successfully developed a machine learning model to predict credit default risk using demographic, financial, and behavioral data from the Home Credit Default Risk dataset.

Achievements:
1. Implemented baseline and advanced models with strong predictive performance.
2. Engineered meaningful features like `debt-to-income ratio`.
3. Used SHAP analysis for interpretability to provide actionable insights.

Future Work:
- Implement XGBoost or LightGBM for further improvements.
- Explore deep learning models if additional computational resources are available.
